In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 用 HuggingFace 官方实现加载 Gemma 4 E4B 作为「参考答案」，用来校验本仓库自实现 Gemma4 的正确性
model_id = "google/gemma-4-E4B"  # HF Hub 上的模型仓库 ID（需已获授权/可访问）
prompt = "Give me a short introduction to large language models."

tokenizer = AutoTokenizer.from_pretrained(model_id)  # 官方分词器
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # 自动权重精度
    device_map="auto",    # 自动分配设备
)
model.eval();  # 评估模式；分号抑制输出
device = next(model.parameters()).device  # 模型所在设备
device

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)  # 编码并搬到模型设备
input_len = inputs["input_ids"].shape[-1]  # prompt 长度，用于截取新生成部分

with torch.inference_mode():  # 推理模式，禁梯度
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,          # 最多生成 200 个新 token
        do_sample=False,             # 贪心解码，结果确定可复现
        pad_token_id=tokenizer.eos_token_id,
    )

# 只解码新生成部分并去除特殊 token
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response.strip())